# Importing Libraries

In [1]:
!pip install mlflow


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import mlflow
import os

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

In [3]:
mlflow.set_experiment("Loan_Status_Experiment")

<Experiment: artifact_location='/Users/shivam13juna/Documents/scaler/mlops/may_2026/may-2026-dsml-mlops/session_8_mlflow/mlruns/1', creation_time=1781056521411, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781056521411, lifecycle_stage='active', name='Loan_Status_Experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
train_df = pd.read_csv('data.csv')
train_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [5]:
train_df.Loan_Status.value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [6]:
# let's binary encode, Gender, Married Loan_Status

train_df['Gender'] = train_df['Gender'].map({'Male': 0, 'Female': 1})
train_df['Married'] = train_df['Married'].map({'No': 0, 'Yes': 1})
train_df['Loan_Status'] = train_df['Loan_Status'].map({'N': 0, 'Y': 1})

In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    float64
 2   Married            611 non-null    float64
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    int64  
dtypes: float64(6), int64(2), object(5)
memory usage: 62.5+ KB


In [8]:
train_df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [9]:
## dropping all the missing values
train_df = train_df.dropna()
train_df.isnull().sum()


# Don't drop features off data off in production.

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

# Let's create simple Features

In [10]:
feature_columns = ['Gender', 'Married', 'ApplicantIncome', 'LoanAmount', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History']
X = train_df[feature_columns]
y = train_df.Loan_Status


In [11]:
mlflow.log_param("feature_columns", feature_columns)

['Gender',
 'Married',
 'ApplicantIncome',
 'LoanAmount',
 'CoapplicantIncome',
 'Loan_Amount_Term',
 'Credit_History']

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=5)

# Let's begin model training

In [13]:
from sklearn.ensemble import RandomForestClassifier

depth = 20000
criteria = 'entropy'
n_esti = 100000

#Will the bias increase or decrease if number of estimators increases?

mlflow.log_param("max_depth", depth)
mlflow.log_param("criterion", criteria)
mlflow.log_param("n_estimators", n_esti)
mlflow.log_param("model_name", "RandomForestClassifier")

#mlflow.log_params({
#	"max_depth": depth,
#	"criterion": criteria,
#	"n_estimators": n_esti
#})

model = RandomForestClassifier(max_depth=depth, random_state=5, criterion=criteria, n_estimators=n_esti)
model.fit(X_train, y_train)

,n_estimators,100000
,criterion,'entropy'
,max_depth,20000
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
from sklearn.metrics import accuracy_score

pred_val = model.predict(X_val)
val_acc_score = accuracy_score(y_val, pred_val)

print(f"Training Accuracy: {model.score(X_train, y_train)}, Validation Accuracy: {val_acc_score}")


Training Accuracy: 1.0, Validation Accuracy: 0.8125


In [15]:
mlflow.log_metric("train_accuracy", model.score(X_train, y_train))
mlflow.log_metric("val_accuracy", val_acc_score)

In [16]:
mlflow.end_run()

# Another efficient way to use MLflow

In [17]:
def mlflow_runs(n_est,max_dep,i):
    with mlflow.start_run():

        model_rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_dep, random_state=5)
        model_rf.fit(X_train, y_train)

        pred_val = model_rf.predict(X_val)
        val_acc=accuracy_score(y_val, pred_val)

        pred_train = model_rf.predict(X_train)
        train_acc=accuracy_score(y_train, pred_train)

        run="hyperparameter_run_"+str(i)
        mlflow.set_tag('mlflow.runName',run)
        mlflow.log_param('n_estimators',n_est)
        mlflow.log_param('max_depth',max_dep)
        mlflow.log_param('model_name','RandomForestClassifier')
        mlflow.log_metric('val_acc',val_acc)
        mlflow.log_metric('train_acc',train_acc)
        mlflow.set_tag('data file','data_new.csv')

        mlflow.sklearn.log_model(model_rf, "model")


mlflow_runs(10,2,1)
mlflow_runs(20,2,2)
mlflow_runs(40,2,3)
mlflow_runs(10,4,4)
mlflow_runs(20,4,5)
mlflow_runs(40,4,6)
mlflow_runs(10,8,7)
mlflow_runs(20,8,8)
mlflow_runs(40,8,9)

2026/06/10 07:48:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 07:48:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/10 07:48:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 07:48:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p